# 63. LoRA Variants Benchmark | LoRA 变体对比项目

**难度：** Hard | **环境：** CPU-first | **标签：** `项目实战`, `LoRA`, `Benchmark` | **目标人群：** LoRA 变体评估与训练工程

---

## 本节导读

LoRA 变体的讨论如果只停在理论层面，很容易陷入“哪个名字更先进”的争论。本节把 LoRA 变体做成 benchmark 项目页：先固定评价口径，再比较不同 rank、alpha、dropout、target modules 和资源消耗，最后输出适合当前预算的方案。


## 前置阅读

**导语：** 先看 LoRA 机制、梯度累积、端到端训练和基础项目页，再做变体 benchmark；这页重点是对比和排序。
- [10. LoRA Tutorial | LoRA 教程](./10_LoRA_Tutorial.ipynb)
- [12. Gradient Accumulation | 梯度累积](./12_Gradient_Accumulation.ipynb)
- [13. End-to-End Fine-Tuning Experiment | 端到端微调实验](./13_End_to_End_Fine_Tuning_Experiment.ipynb)
- [60. LoRA Fine-Tuning Project | LoRA 微调项目](./60_LoRA_Fine_Tuning_Project.ipynb)


### Step 1: 定义比较维度
先回答一个问题：这次 benchmark 是比较训练效率、参数占比、显存，还是最终验证集表现？

- 固定底座模型、数据集、batch size、seq len、优化器和训练步数。
- 明确候选 LoRA 变体，例如不同 rank、alpha、dropout、target modules 或初始化策略。
- 统一记录 train loss、val loss、step time、peak memory、可训练参数量和参数占比。
- 先设定预算边界，再决定哪个变体值得进入后续项目。


#### 图解：10-60 如何收束到 63 LoRA Benchmark

`63` 把 LoRA 机制和项目经验收成一张统一的 benchmark 表。

```text
10 LoRA          target modules / rank / alpha / dropout
      │
12 Accumulation  micro batch -> effective batch
      │
13 E2E report     train loss / val loss / step time / memory
      │
60 LoRA project   baseline vs LoRA artifact and ledger
      │
      ▼
63 LoRA bench     variant ranking + budget-aware recommendation
```

项目页最小产物：


In [ ]:
from typing import Dict, List


In [ ]:
# TODO: 完成 LoRA 变体评分、排序和项目推荐
# 目标：把不同 LoRA 变体转成统一的 benchmark 结果

def score_lora_variant(variant):
    # ==========================================
    # TODO 1: 为单个变体打分
    # 提示：把 train loss、val loss、step time 和 memory 转成一个可比较的分数。
    # ==========================================
    return {
        'name': variant.get('name', 'variant'),
        'score': 0.0,
        'memory_mb': variant.get('memory_mb', 0),
    }

def rank_lora_variants(variants):
    # ==========================================
    # TODO 2: 对变体排序
    # 提示：优先比较综合 score，再记录资源消耗。
    # ==========================================
    return []

def recommend_lora_variant(variants, memory_budget_mb):
    # ==========================================
    # TODO 3: 给出预算内推荐
    # 提示：在显存预算内选择综合表现最优的变体。
    # ==========================================
    return {
        'recommended_name': None,
        'within_budget': False,
        'memory_budget_mb': memory_budget_mb,
    }


In [ ]:
# 测试你的实现
def test_lora_benchmark_template():
    try:
        variants = [
            {'name': 'rank4', 'train_loss': 1.2, 'val_loss': 1.4, 'step_time_ms': 90, 'memory_mb': 1100},
            {'name': 'rank8', 'train_loss': 1.1, 'val_loss': 1.2, 'step_time_ms': 110, 'memory_mb': 1350},
        ]
        scored = score_lora_variant(variants[0])
        assert 'score' in scored and 'name' in scored, '单个变体评分结果不完整！'
        ranked = rank_lora_variants(variants)
        assert isinstance(ranked, list), '排序结果必须是列表！'
        decision = recommend_lora_variant(variants, memory_budget_mb=1200)
        assert 'recommended_name' in decision and 'within_budget' in decision, '推荐结果字段缺失！'
        print('测试通过：LoRA 变体 benchmark 模板结构正常。')
    except Exception as exc:
        print(f'测试未通过：{exc}')

test_lora_benchmark_template()


🛑 **STOP HERE** 🛑


## 参考代码与解析

### 代码


In [ ]:
# TODO 1: 为单个变体打分
def score_lora_variant(variant):
    train_loss = variant.get('train_loss', 0.0)
    val_loss = variant.get('val_loss', 0.0)
    step_time_ms = variant.get('step_time_ms', 0)
    memory_mb = variant.get('memory_mb', 0)

    score = val_loss + 0.01 * train_loss + 0.001 * step_time_ms + 0.0001 * memory_mb
    return {
        'name': variant.get('name', 'variant'),
        'score': score,
        'memory_mb': memory_mb,
    }

# TODO 2: 对变体排序
def rank_lora_variants(variants):
    scored = [score_lora_variant(variant) for variant in variants]
    return sorted(scored, key=lambda item: item['score'])

# TODO 3: 给出预算内推荐
def recommend_lora_variant(variants, memory_budget_mb):
    feasible = [variant for variant in variants if variant.get('memory_mb', 10**9) <= memory_budget_mb]
    if not feasible:
        return {
            'recommended_name': None,
            'within_budget': False,
            'memory_budget_mb': memory_budget_mb,
        }

    ranked = rank_lora_variants(feasible)
    best = ranked[0]
    return {
        'recommended_name': best['name'],
        'within_budget': True,
        'memory_budget_mb': memory_budget_mb,
    }


### 解析

**1. TODO 1: 为单个变体打分**
- **实现方式**：把验证损失、训练损失、步时和显存统一折算成一个分数。
- **关键点**：benchmark 的核心是统一口径，否则不同变体没法横向比较。
- **项目意义**：让 LoRA 变体的讨论从“概念优劣”转成“可量化优劣”。

**2. TODO 2: 对变体排序**
- **实现方式**：先给每个变体打分，再按分数排序，得到明确的推荐顺序。
- **关键点**：排序函数要稳定，才能支撑后续项目结论和复现。
- **项目意义**：这一步把多个候选方案收成一张 benchmark 排名表。

**3. TODO 3: 给出预算内推荐**
- **实现方式**：先过滤掉超出显存预算的候选，再从剩余方案里挑综合分数最优者。
- **关键点**：工程决策必须尊重预算，不然推荐结论没有落地价值。
- **项目意义**：把 benchmark 结果转换为真实可执行的方案选择。
